# Model 1A, CatBoost Experiment 06: Combined 2019 and 2023 Training

This append-only post-test sensitivity experiment copies the frozen CatBoost 1A-04 design. It changes only the training population: the model is fit on combined 2019 and 2023 rows instead of 2019 alone. The 41-field allowlist, 24-hour rotation mask, classifier settings, and 0.31 operating threshold remain unchanged.

The 2024 outcomes were already examined in the official final evaluation. This experiment therefore cannot replace that result or select a new model. It measures whether routine retraining with the completed 2023 development year changes the frozen design's 2024 performance.

In [1]:
from pathlib import Path
from time import perf_counter

import catboost
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score,
)

AIRPORT = "JFK"
TRAIN_YEARS = (2019, 2023)
TEST_YEAR = 2024
TARGET = "DepDel15"
FROZEN_THRESHOLD = 0.31
RANDOM_STATE = 42
N_JOBS = 4

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
print(f"CatBoost {catboost.__version__}")

CatBoost 1.2.10


## Load and validate the established feature datasets

Each year is validated before concatenation. The rotation, airport-wide backlog, and same-airline backlog datasets must contain identical target rows within a year.

In [2]:
def find_project_root(start: Path) -> Path:
    required = [
        Path("data/features") / f"{AIRPORT}_{year}_departures_{suffix}.csv"
        for year in (*TRAIN_YEARS, TEST_YEAR)
        for suffix in (
            "rotation_full_history", "backlog_w60", "backlog_airline_w60",
        )
    ]
    for candidate in (start, *start.parents):
        if all((candidate / path).is_file() for path in required):
            return candidate
    raise FileNotFoundError(f"Could not locate required datasets: {required}")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
FEATURE_DIR = PROJECT_ROOT / "data/features"
years = (*TRAIN_YEARS, TEST_YEAR)
rotation_frames = {
    year: pd.read_csv(FEATURE_DIR / f"{AIRPORT}_{year}_departures_rotation_full_history.csv", low_memory=False)
    for year in years
}
backlog_frames = {
    year: pd.read_csv(FEATURE_DIR / f"{AIRPORT}_{year}_departures_backlog_w60.csv", low_memory=False)
    for year in years
}
airline_frames = {
    year: pd.read_csv(FEATURE_DIR / f"{AIRPORT}_{year}_departures_backlog_airline_w60.csv", low_memory=False)
    for year in years
}

ROTATION_FEATURES = [
    "ROTATION_STATUS", "ROTATION_MATCH_FOUND", "ROTATION_INBOUND_ORIGIN",
    "ROTATION_SCHEDULED_TURN_MINUTES", "ROTATION_INBOUND_ARRIVED_BY_CUTOFF",
    "ROTATION_INBOUND_NOT_ARRIVED_BY_CUTOFF", "ROTATION_INBOUND_OVERDUE_MINUTES",
    "ROTATION_LOG_INBOUND_OVERDUE_MINUTES", "ROTATION_ACTUAL_TURN_MINUTES",
    "ROTATION_LOG_ACTUAL_TURN_MINUTES", "ROTATION_INBOUND_ARR_DELAY",
    "ROTATION_INBOUND_DELAYED_15", "ROTATION_LOG_SCHEDULED_TURN_MINUTES",
]
BACKLOG_FEATURES = [
    "BACKLOG_W60_PENDING_COUNT", "BACKLOG_W60_COMPLETED_COUNT",
    "BACKLOG_W60_MEAN_DEP_DELAY",
]
AIRLINE_BACKLOG_FEATURES = [
    "AIRLINE_BACKLOG_W60_PENDING_COUNT", "AIRLINE_BACKLOG_W60_COMPLETED_COUNT",
    "AIRLINE_BACKLOG_W60_MEAN_DEP_DELAY", "AIRLINE_BACKLOG_W60_DELAY_RATE",
    "AIRLINE_BACKLOG_W60_PENDING_SHARE",
]
IDENTITY_COLUMNS = [
    "FlightDate", "Reporting_Airline", "Flight_Number_Reporting_Airline",
    "Origin", "Dest", "CRSDepTime", "Tail_Number", TARGET,
]


def combine_year(year):
    rotation = rotation_frames[year].copy()
    backlog = backlog_frames[year].copy()
    airline = airline_frames[year].copy()
    assert len(rotation) == len(backlog) == len(airline)
    pd.testing.assert_frame_equal(rotation[IDENTITY_COLUMNS], backlog[IDENTITY_COLUMNS], check_dtype=False)
    pd.testing.assert_frame_equal(rotation[IDENTITY_COLUMNS], airline[IDENTITY_COLUMNS], check_dtype=False)
    for frame in (rotation, backlog, airline):
        frame["FlightDate"] = pd.to_datetime(frame["FlightDate"], errors="raise")
        assert frame["FlightDate"].dt.year.eq(year).all()
        assert frame["Origin"].eq(AIRPORT).all()
    match = pd.to_numeric(rotation["ROTATION_MATCH_FOUND"], errors="raise")
    arrived = pd.to_numeric(rotation["ROTATION_INBOUND_ARRIVED_BY_CUTOFF"], errors="raise")
    not_arrived = pd.to_numeric(rotation["ROTATION_INBOUND_NOT_ARRIVED_BY_CUTOFF"], errors="raise")
    assert match.eq(arrived + not_arrived).all()
    actual_only = [
        "ROTATION_ACTUAL_TURN_MINUTES", "ROTATION_INBOUND_ARR_DELAY",
        "ROTATION_INBOUND_DELAYED_15",
    ]
    assert not rotation.loc[arrived.eq(0), actual_only].notna().any().any()
    combined = rotation.copy()
    for column in BACKLOG_FEATURES:
        combined[column] = backlog[column].to_numpy()
    for column in AIRLINE_BACKLOG_FEATURES:
        combined[column] = airline[column].to_numpy()
    return combined


combined_frames = {year: combine_year(year) for year in years}
pd.DataFrame([
    {"year": year, "rows": len(combined_frames[year]), "delay_rate": combined_frames[year][TARGET].mean()}
    for year in years
]).set_index("year")

,rows,delay_rate
year,,
2019,107430,0.1877
2023,109983,0.2364
2024,104715,0.2038


## Fit the unchanged CatBoost 1A-04 design

The two training years are placed in chronological order. No temporal search, feature comparison, calibration, or threshold selection is repeated.

In [3]:
CATEGORICAL_FEATURES = [
    "Month", "DayOfWeek", "Reporting_Airline", "Dest",
    "ROTATION_STATUS", "ROTATION_INBOUND_ORIGIN",
]
BASE_NUMERIC_FEATURES = [
    "CRSDepTime", "CRSArrTime", "CRSElapsedTime", "Distance",
    "ASPM_PREVIOUS_SCHEDULED_DEPARTURES", "ASPM_PREVIOUS_SCHEDULED_ARRIVALS",
    "ASPM_CURRENT_SCHEDULED_DEPARTURES", "ASPM_CURRENT_SCHEDULED_ARRIVALS",
    "ASPM_NEXT_SCHEDULED_DEPARTURES", "ASPM_NEXT_SCHEDULED_ARRIVALS",
    "HourlyDewPointTemperature", "HourlyDryBulbTemperature",
    "HourlyPrecipitation", "HourlyRelativeHumidity", "HourlyVisibility", "HourlyWindSpeed",
]
ROTATION_NUMERIC = [field for field in ROTATION_FEATURES if field not in CATEGORICAL_FEATURES]
NUMERIC_FEATURES = [
    *BASE_NUMERIC_FEATURES, *ROTATION_NUMERIC,
    *BACKLOG_FEATURES, *AIRLINE_BACKLOG_FEATURES,
]
FEATURES = [*CATEGORICAL_FEATURES, *NUMERIC_FEATURES]
assert len(FEATURES) == 41 and len(CATEGORICAL_FEATURES) == 6 and len(NUMERIC_FEATURES) == 35


def prepare(source):
    frame = source.copy()
    long_turn = pd.to_numeric(frame["ROTATION_SCHEDULED_TURN_MINUTES"], errors="coerce").gt(24 * 60)
    frame.loc[long_turn, ROTATION_FEATURES] = np.nan
    frame.loc[long_turn, "ROTATION_STATUS"] = "LONG_TURN_EXCLUDED"
    required = list(dict.fromkeys(["FlightDate", "CRSDepTime", TARGET, *FEATURES]))
    frame = frame[required].sort_values(["FlightDate", "CRSDepTime"], kind="stable").reset_index(drop=True)
    frame[TARGET] = pd.to_numeric(frame[TARGET], errors="raise").astype(int)
    for column in CATEGORICAL_FEATURES:
        frame[column] = frame[column].astype("string").fillna("MISSING").astype(str)
    for column in NUMERIC_FEATURES:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    return frame


prepared = {year: prepare(combined_frames[year]) for year in years}
train = pd.concat([prepared[year] for year in TRAIN_YEARS], ignore_index=True)
train = train.sort_values(["FlightDate", "CRSDepTime"], kind="stable").reset_index(drop=True)
test = prepared[TEST_YEAR]
assert train["FlightDate"].max() < test["FlightDate"].min()

fit_start = perf_counter()
model = CatBoostClassifier(
    iterations=683, depth=6, learning_rate=0.03, l2_leaf_reg=3,
    random_strength=1, loss_function="Logloss", eval_metric="PRAUC:type=Classic",
    random_seed=RANDOM_STATE, thread_count=N_JOBS,
    allow_writing_files=False, verbose=False,
)
model.fit(train[FEATURES], train[TARGET], cat_features=CATEGORICAL_FEATURES, verbose=False)
fit_seconds = perf_counter() - fit_start
predict_start = perf_counter()
probabilities = model.predict_proba(test[FEATURES])[:, 1]
predict_seconds = perf_counter() - predict_start

print(f"Training rows: {len(train):,}; delay rate: {train[TARGET].mean():.4f}")
print(f"Fit: {fit_seconds:,.1f}s; 2024 prediction: {predict_seconds:,.1f}s")

Training rows: 217,413; delay rate: 0.2123
Fit: 51.0s; 2024 prediction: 0.1s


## Post-test comparison

The official row is copied from the final 2019-only evaluation. The combined-training row uses the same 2024 outcomes and is reported only as a retraining sensitivity.

In [4]:
def ranking_metrics(y_true, probabilities):
    return {
        "average_precision": average_precision_score(y_true, probabilities),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "brier_score": brier_score_loss(y_true, probabilities),
    }


def operating_metrics(y_true, probabilities, threshold):
    predictions = (np.asarray(probabilities) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, predictions),
        "balanced_accuracy": balanced_accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "mcc": matthews_corrcoef(y_true, predictions),
        "predicted_positive_rate": predictions.mean(),
        "tn": tn, "fp": fp, "fn": fn, "tp": tp,
    }

combined_ranking = ranking_metrics(test[TARGET], probabilities)
combined_default = operating_metrics(test[TARGET], probabilities, 0.50)
combined_frozen = operating_metrics(test[TARGET], probabilities, FROZEN_THRESHOLD)

official_ranking = {"average_precision": 0.7250, "roc_auc": 0.8517, "brier_score": 0.0983}
official_frozen = {
    "threshold": 0.31, "accuracy": 0.8670, "balanced_accuracy": 0.7530,
    "precision": 0.7246, "recall": 0.5605, "f1": 0.6321, "mcc": 0.5593,
}

ranking_comparison = pd.DataFrame([
    {"training": "2019 only — official final", **official_ranking},
    {"training": "2019 + 2023 — post-test", **combined_ranking},
]).set_index("training")
for metric in official_ranking:
    ranking_comparison.loc["2019 + 2023 — post-test", f"{metric}_change"] = (
        combined_ranking[metric] - official_ranking[metric]
    )

operating_results = pd.DataFrame([
    {"training": "2019 only — official final", "threshold_policy": "Frozen", **official_frozen},
    {"training": "2019 + 2023 — post-test", "threshold_policy": "Default", **combined_default},
    {"training": "2019 + 2023 — post-test", "threshold_policy": "Frozen", **combined_frozen},
]).set_index(["training", "threshold_policy"])

display(ranking_comparison)
display(operating_results)

,average_precision,roc_auc,brier_score,average_precision_change,roc_auc_change,brier_score_change
training,,,,,,
2019 only — official final,0.7250,0.8517,0.0983,NaN,NaN,NaN
2019 + 2023 — post-test,0.7313,0.8553,0.0965,0.0063,0.0036,-0.0018


threshold  accuracy  \
training                   threshold_policy                        
2019 only — official final Frozen               0.3100    0.8670   
2019 + 2023 — post-test    Default              0.5000    0.8749   
                           Frozen               0.3100    0.8622   

                                             balanced_accuracy  precision  \
training                   threshold_policy                                 
2019 only — official final Frozen                       0.7530     0.7246   
2019 + 2023 — post-test    Default                      0.7259     0.8429   
                           Frozen                       0.7641     0.6854   

                                             recall     f1    mcc  \
training                   threshold_policy                         
2019 only — official final Frozen            0.5605 0.6321 0.5593   
2019 + 2023 — post-test    Default           0.4745 0.6072 0.5711   
                           Frozen            0.5986 0.6391 0.5563   

                                             predicted_positive_rate  \
training                   threshold_policy                            
2019 only — official final Frozen                                NaN   
2019 + 2023 — post-test    Default                            0.1147   
                           Frozen                             0.1780   

                                                     tn         fp  \
training                   threshold_policy                          
2019 only — official final Frozen                   NaN        NaN   
2019 + 2023 — post-test    Default          81,483.0000 1,888.0000   
                           Frozen           77,505.0000 5,866.0000   

                                                     fn          tp  
training                   threshold_policy                          
2019 only — official final Frozen                   NaN         NaN  
2019 + 2023 — post-test    Default          11,216.0000 10,128.0000  
                           Frozen            8,567.0000 12,777.0000

## Interpretation rule

This experiment changes only the training population. Its result does not replace CatBoost 1A-04 as the official selected experiment and does not reopen model or threshold selection. The retrospective aircraft-assignment limitation remains unchanged.